In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/raw/reels_attention_span_dirty.csv")


In [3]:
df.head()

,user_id,age,reels_watch_time_hours,daily_screen_time_hours,sleep_hours,attention_span_score,focus_level,task_completion_rate,stress_level,platform
0,1,21.0,1.86,7.34,8.95,4.99,5.09,45.44,Medium,TikTok
1,2,34.0,3.03,5.78,5.78,3.73,8.25,50.17,High,TikTok
2,3,43.0,3.96,9.62,5.46,5.48,8.96,43.27,Low,TikTok
3,4,29.0,3.56,7.25,5.09,5.54,9.77,64.99,Low,Instagram Reels
4,5,25.0,4.58,NaN,NaN,8.07,4.11,89.28,Low,Instagram Reels


In [4]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   user_id                  12000 non-null  int64  
 1   age                      11822 non-null  float64
 2   reels_watch_time_hours   11786 non-null  object 
 3   daily_screen_time_hours  11856 non-null  object 
 4   sleep_hours              11814 non-null  object 
 5   attention_span_score     11834 non-null  float64
 6   focus_level              12000 non-null  float64
 7   task_completion_rate     11845 non-null  float64
 8   stress_level             12000 non-null  object 
 9   platform                 12000 non-null  object 
dtypes: float64(4), int64(1), object(5)
memory usage: 937.6+ KB


In [5]:
df.describe()

,user_id,age,attention_span_score,focus_level,task_completion_rate
count,12000.000000,11822.000000,11834.000000,12000.000000,11845.000000
mean,5904.611500,29.676282,6.783389,5.511036,69.867918
std,3411.662456,8.662818,11.125716,2.590792,17.288720
min,1.000000,15.000000,-10.000000,0.954000,37.934000
25%,2948.750000,22.000000,3.250000,3.297500,54.700000
50%,5899.500000,30.000000,5.545000,5.500000,69.920000
75%,8860.250000,37.000000,7.830000,7.780000,84.870000
max,11820.000000,44.000000,110.000000,10.615000,105.686000


In [6]:
df.duplicated().sum()

np.int64(129)

In [7]:
q1 = df["attention_span_score"].quantile(0.25)
q3 = df["attention_span_score"].quantile(0.75)
iqr = q3 - q1
up = 1.5*iqr + q3
low = q1 - 1.5*iqr

outliers = df[(df["attention_span_score"] > up ) | (df["attention_span_score"] < low)]

In [8]:
df2 = df.drop_duplicates()

In [9]:
df2.duplicated().sum()

np.int64(0)

# Data Cleaning

In [10]:

# 2. COPY

df2 = df.copy()


# 3. FIX NUMERIC COLUMNS

numeric_cols = [
    'age',
    'reels_watch_time_hours',
    'daily_screen_time_hours',
    'sleep_hours',
    'attention_span_score',
    'focus_level',
    'task_completion_rate'
]

# Convert object → numeric
df2[numeric_cols] = df2[numeric_cols].apply(pd.to_numeric, errors='coerce')


# 4. CLEAN CATEGORICAL COLUMNS


# Normalize text
df2['platform'] = df2['platform'].astype(str).str.strip().str.lower()
df2['stress_level'] = df2['stress_level'].astype(str).str.strip().str.title()

# Map platform variations
platform_mapping = {
    'insta': 'Instagram Reels',
    'instagram': 'Instagram Reels',
    'instagram reels': 'Instagram Reels',
    'ig': 'Instagram Reels',

    'youtube': 'Youtube Shorts',
    'youtube shorts': 'Youtube Shorts',
    'yt': 'Youtube Shorts',

    'tiktok': 'Tiktok',
    'tik tok': 'Tiktok'
}

df2['platform'] = df2['platform'].map(platform_mapping)

# Handle unknowns
df2['platform'].fillna('Unknown', inplace=True)

# Fix stress level categories
valid_stress = ['Low', 'Medium', 'High']
df2['stress_level'] = df2['stress_level'].where(df2['stress_level'].isin(valid_stress))
df2['stress_level'].fillna('Unknown', inplace=True)

# 5. REMOVE INVALID VALUES


# Age
df2 = df2[df2['age'].between(10, 100)]

# Time constraints
df2 = df2[df2['daily_screen_time_hours'].between(0, 24)]
df2 = df2[df2['sleep_hours'].between(0, 24)]
df2 = df2[df2['reels_watch_time_hours'] >= 0]

# Logical constraint
df2 = df2[df2['reels_watch_time_hours'] <= df2['daily_screen_time_hours']]

# Scores (1–10 scale)
df2 = df2[df2['attention_span_score'].between(1, 10)]
df2 = df2[df2['focus_level'].between(1, 10)]

# Percentage
df2 = df2[df2['task_completion_rate'].between(0, 100)]


# 6. HANDLE MISSING VALUES


# Numeric (excluding user_id)
numeric_features = df2.select_dtypes(include=[np.number]).columns.drop('user_id')

df2[numeric_features] = df2[numeric_features].fillna(
    df2[numeric_features].median()
)


# 7. OUTLIER HANDLING (SELECTIVE)


def clip_iqr(df, col):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    df[col] = df[col].clip(lower, upper)

outlier_cols = [
    'reels_watch_time_hours',
    'daily_screen_time_hours',
    'sleep_hours'
]

for col in outlier_cols:
    clip_iqr(df2, col)


# 8. FINAL CLEANUP

df2.drop_duplicates(inplace=True)
df2.reset_index(drop=True, inplace=True)


# 9. VALIDATION

print("Final Shape:", df2.shape)
print("\nMissing Values:\n", df2.isnull().sum())
print("\nSummary:\n", df2.describe())

df2.head()

Final Shape: (8926, 10)

Missing Values:
 user_id                    0
age                        0
reels_watch_time_hours     0
daily_screen_time_hours    0
sleep_hours                0
attention_span_score       0
focus_level                0
task_completion_rate       0
stress_level               0
platform                   0
dtype: int64

Summary:
             user_id          age  reels_watch_time_hours  \
count   8926.000000  8926.000000             8926.000000   
mean    5876.545037    29.573157                3.022674   
std     3405.444388     8.658085                1.544532   
min        1.000000    15.000000                0.500000   
25%     2935.250000    22.000000                1.700000   
50%     5860.500000    30.000000                2.910000   
75%     8803.750000    37.000000                4.290000   
max    11820.000000    44.000000                6.000000   

       daily_screen_time_hours  sleep_hours  attention_span_score  \
count              8926.000000  89

/var/folders/_l/3j_1f6pn76g_tm97wf9fg6kw0000gn/T/ipykernel_13434/1157058400.py:47: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df2['platform'].fillna('Unknown', inplace=True)
/var/folders/_l/3j_1f6pn76g_tm97wf9fg6kw0000gn/T/ipykernel_13434/1157058400.py:52: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values alw

,user_id,age,reels_watch_time_hours,daily_screen_time_hours,sleep_hours,attention_span_score,focus_level,task_completion_rate,stress_level,platform
0,1,21.0,1.86,7.34,8.95,4.99,5.09,45.44,Medium,Tiktok
1,2,34.0,3.03,5.78,5.78,3.73,8.25,50.17,High,Tiktok
2,3,43.0,3.96,9.62,5.46,5.48,8.96,43.27,Low,Tiktok
3,4,29.0,3.56,7.25,5.09,5.54,9.77,64.99,Low,Instagram Reels
4,6,22.0,2.04,16.43,7.54,2.11,6.35,88.73,Medium,Youtube Shorts


In [11]:
df2.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8926 entries, 0 to 8925
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   user_id                  8926 non-null   int64  
 1   age                      8926 non-null   float64
 2   reels_watch_time_hours   8926 non-null   float64
 3   daily_screen_time_hours  8926 non-null   float64
 4   sleep_hours              8926 non-null   float64
 5   attention_span_score     8926 non-null   float64
 6   focus_level              8926 non-null   float64
 7   task_completion_rate     8926 non-null   float64
 8   stress_level             8926 non-null   object 
 9   platform                 8926 non-null   object 
dtypes: float64(7), int64(1), object(2)
memory usage: 697.5+ KB
